# RL-LLM: Reinforcement Learning for Language Model Training

This notebook implements a hierarchical RL-based language model using PPO (Proximal Policy Optimization).

**Week 5-6 Implementation:**
- Multi-component reward system (fluency, coherence, task completion, safety)
- Task-specific environments (Q&A, Conversation)
- Hierarchical policy (High-level: intentions, Low-level: tokens)
- Hierarchical PPO training

**Goal:** Generate coherent text for simple tasks like question-answering and conversation

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install torch>=2.0.0
!pip install transformers>=4.30.0
!pip install datasets>=2.14.0
!pip install numpy>=1.24.0
!pip install tqdm>=4.65.0

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA version: 12.6
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.17 GB


## 2. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import random
import math
import re
import os
from typing import Dict, List, Tuple, Optional, Any
from io import StringIO
import sys
import signal
from contextlib import contextmanager

## 3. Utility Functions

In [ ]:
# Training utilities
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    """Get appropriate device (CUDA if available)"""
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set seed for reproducibility
set_seed(42)
device = get_device()
print(f"Using device: {device}")

Using device: cuda


## 4. Dataset Loader Classes

In [ ]:
class TimeoutException(Exception):
    """Custom exception for code execution timeout"""
    pass

@contextmanager
def time_limit(seconds):
    """Context manager for enforcing execution timeout"""
    def signal_handler(signum, frame):
        raise TimeoutException("Timed out!")

    signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)

In [ ]:
# CELL 4 (REPLACEMENT): Programming Dataset Loaders

class HumanEvalDataset:
    """HumanEval dataset for code generation"""

    def __init__(self, split: str = 'test', use_subset: Optional[int] = None):
        print("Loading HumanEval dataset...")
        try:
            from datasets import load_dataset
            dataset = load_dataset("openai_humaneval", split=split)
        except Exception as e:
            print(f"Error loading HumanEval: {e}")
            dataset = []

        self.problems = []
        for example in dataset:
            problem = {
                'task_id': example['task_id'],
                'prompt': example['prompt'],
                'test': example['test'],
                'entry_point': example['entry_point']
            }
            self.problems.append(problem)

        if use_subset is not None:
            self.problems = self.problems[:use_subset]

        print(f"Loaded {len(self.problems)} code problems")

    def get_random_problem(self) -> Dict:
        return random.choice(self.problems)

    def get_problem_by_id(self, task_id: str) -> Optional[Dict]:
        for p in self.problems:
            if p['task_id'] == task_id:
                return p
        return None

    def evaluate_code(self, code: str, problem: Dict) -> Tuple[bool, int, Optional[str]]:
        """Execute code and check if tests pass"""
        try:
            # Combine generated code with test cases
            full_code = code + '\n\n' + problem['test'] + '\n\n'
            full_code += f"check({problem['entry_point']})\n"

            # Execute with timeout (5 seconds)
            old_stdout = sys.stdout
            old_stderr = sys.stderr
            redirected_output = StringIO()
            sys.stdout = redirected_output
            sys.stderr = redirected_output

            try:
                exec(full_code, {})
                sys.stdout = old_stdout
                sys.stderr = old_stderr
                return (True, 1, None)
            except Exception as e:
                sys.stdout = old_stdout
                sys.stderr = old_stderr
                return (False, 0, str(e))
        except Exception as e:
            return (False, 0, str(e))

    def compute_reward(self, code: str, problem: Dict) -> float:
        """Compute reward from code execution"""
        passed, total, error = self.evaluate_code(code, problem)

        if passed:
            return 10.0
        elif error and "SyntaxError" in str(error):
            return -5.0
        elif error and "Timeout" in str(error):
            return -3.0
        else:
            return -1.0

    def __len__(self):
        return len(self.problems)

    def __getitem__(self, idx):
        return self.problems[idx]


class TheStackDataset:
    """The Stack dataset - filtered for Python code"""

    def __init__(self, language: str = 'python', num_samples: int = 1000):
        print(f"Loading The Stack dataset ({language})...")
        try:
            from datasets import load_dataset
            # Load a streaming subset
            dataset = load_dataset(
                "bigcode/the-stack-dedup",
                data_dir=f"data/{language}",
                split="train",
                streaming=True
            )

            # Take first num_samples
            self.problems = []
            for i, example in enumerate(dataset):
                if i >= num_samples:
                    break

                # Extract code snippet
                code = example['content']

                # Create a "complete the function" task
                # Split code into prompt and completion
                lines = code.split('\n')
                if len(lines) > 10:
                    split_point = len(lines) // 2
                    prompt = '\n'.join(lines[:split_point])
                    expected = '\n'.join(lines[split_point:])

                    self.problems.append({
                        'task_id': f'stack_{i}',
                        'prompt': prompt,
                        'expected': expected,
                        'full_code': code
                    })

            print(f"Loaded {len(self.problems)} code samples")
        except Exception as e:
            print(f"Error loading The Stack: {e}")
            print("You may need to authenticate with HuggingFace Hub")
            self.problems = []

    def get_random_problem(self) -> Dict:
        if not self.problems:
            return {'task_id': 'empty', 'prompt': 'def hello():', 'expected': '\n    return "world"'}
        return random.choice(self.problems)

    def compute_reward(self, generated: str, problem: Dict) -> float:
        """Reward based on similarity to expected completion"""
        expected = problem.get('expected', '')

        # Simple token overlap metric
        gen_tokens = set(generated.split())
        exp_tokens = set(expected.split())

        if not exp_tokens:
            return -1.0

        overlap = len(gen_tokens & exp_tokens) / len(exp_tokens)

        # Reward based on overlap
        if overlap > 0.8:
            return 10.0
        elif overlap > 0.5:
            return 5.0
        elif overlap > 0.3:
            return 2.0
        else:
            return -1.0

    def __len__(self):
        return len(self.problems)

    def __getitem__(self, idx):
        return self.problems[idx]


class CodeChainDataset:
    """CodeChain dataset for chain-of-thought code reasoning"""

    def __init__(self, num_samples: int = 500):
        print("Loading CodeChain dataset...")
        try:
            from datasets import load_dataset
            dataset = load_dataset("Elfsong/CodeChain", split="train")

            self.problems = []
            for i, example in enumerate(dataset):
                if i >= num_samples:
                    break

                problem = {
                    'task_id': f'codechain_{i}',
                    'prompt': example.get('question', ''),
                    'solution': example.get('solution', ''),
                    'chain': example.get('chain_of_thought', '')
                }
                self.problems.append(problem)

            print(f"Loaded {len(self.problems)} CodeChain problems")
        except Exception as e:
            print(f"Error loading CodeChain: {e}")
            self.problems = []

    def get_random_problem(self) -> Dict:
        if not self.problems:
            return {'task_id': 'empty', 'prompt': 'Write a function to add two numbers'}
        return random.choice(self.problems)

    def compute_reward(self, generated: str, problem: Dict) -> float:
        """Reward based on solution similarity"""
        solution = problem.get('solution', '')

        # Check if key elements of solution appear in generated code
        if not solution:
            return 0.0

        # Simple substring matching
        if solution.lower() in generated.lower():
            return 10.0

        # Partial credit for similar tokens
        gen_tokens = set(generated.lower().split())
        sol_tokens = set(solution.lower().split())

        if sol_tokens:
            overlap = len(gen_tokens & sol_tokens) / len(sol_tokens)
            return overlap * 10.0 - 2.0

        return -1.0

    def __len__(self):
        return len(self.problems)

    def __getitem__(self, idx):
        return self.problems[idx]


class RedPajamaCodeDataset:
    """RedPajama dataset - code subset"""

    def __init__(self, num_samples: int = 1000):
        print("Loading RedPajama Code dataset...")
        try:
            from datasets import load_dataset
            # Load GitHub subset of RedPajama
            dataset = load_dataset(
                "togethercomputer/RedPajama-Data-1T-Sample",
                split="train",
                streaming=True
            )

            self.problems = []
            count = 0

            for example in dataset:
                if count >= num_samples:
                    break

                # Filter for code content (GitHub source)
                if example.get('meta', {}).get('redpajama_set_name') == 'RedPajamaGithub':
                    text = example['text']

                    # Create completion task
                    if len(text) > 100:
                        split_point = len(text) // 2
                        prompt = text[:split_point]
                        expected = text[split_point:]

                        self.problems.append({
                            'task_id': f'redpajama_{count}',
                            'prompt': prompt,
                            'expected': expected
                        })
                        count += 1

            print(f"Loaded {len(self.problems)} RedPajama code samples")
        except Exception as e:
            print(f"Error loading RedPajama: {e}")
            self.problems = []

    def get_random_problem(self) -> Dict:
        if not self.problems:
            return {'task_id': 'empty', 'prompt': 'import numpy as np\n\ndef '}
        return random.choice(self.problems)

    def compute_reward(self, generated: str, problem: Dict) -> float:
        """Reward based on completion quality"""
        expected = problem.get('expected', '')

        # Token overlap
        gen_tokens = set(generated.split())
        exp_tokens = set(expected.split())

        if not exp_tokens:
            return -1.0

        overlap = len(gen_tokens & exp_tokens) / len(exp_tokens)

        if overlap > 0.7:
            return 10.0
        elif overlap > 0.4:
            return 5.0
        else:
            return -1.0

    def __len__(self):
        return len(self.problems)

    def __getitem__(self, idx):
        return self.problems[idx]


# Dataset factory function
def create_code_dataset(dataset_type: str = 'humaneval', **kwargs):
    """
    Factory function to create code datasets

    Args:
        dataset_type: 'humaneval', 'stack', 'codechain', or 'redpajama'
        **kwargs: Additional arguments for dataset constructor
    """
    if dataset_type == 'humaneval':
        return HumanEvalDataset(**kwargs)
    elif dataset_type == 'stack':
        return TheStackDataset(**kwargs)
    elif dataset_type == 'codechain':
        return CodeChainDataset(**kwargs)
    elif dataset_type == 'redpajama':
        return RedPajamaCodeDataset(**kwargs)
    else:
        raise ValueError(f"Unknown dataset type: {dataset_type}")


# Test datasets
print("\n" + "="*70)
print("Testing Dataset Loaders")
print("="*70)

# Test HumanEval
try:
    humaneval = create_code_dataset('humaneval', use_subset=5)
    sample = humaneval.get_random_problem()
    print(f"\nHumanEval sample:")
    print(f"  Task: {sample['task_id']}")
    print(f"  Prompt: {sample['prompt'][:100]}...")
except Exception as e:
    print(f"HumanEval loading failed: {e}")

print("\n✓ Dataset loaders ready!")


Testing Dataset Loaders
Loading HumanEval dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loaded 5 code problems

HumanEval sample:
  Task: HumanEval/0
  Prompt: from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
  ...

✓ Dataset loaders ready!


In [ ]:
# NEW CELL 5: Code Generation Environment

class CodeGenerationEnvironment:
    """Specialized RL environment for code generation tasks"""

    def __init__(self, tokenizer, dataset, max_length: int = 512):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.max_length = max_length

        self.current_problem = None
        self.current_sequence = []
        self.step_count = 0
        self.done = False

        self.eos_token_id = tokenizer.eos_token_id

    def reset(self) -> Dict:
        """Reset with a random code problem"""
        self.current_problem = self.dataset.get_random_problem()

        # Tokenize the prompt
        prompt = self.current_problem['prompt']
        token_ids = self.tokenizer.encode(prompt, return_tensors='pt')[0]

        self.current_sequence = token_ids.tolist()
        self.step_count = 0
        self.done = False

        return self.get_state()

    def step(self, action: int) -> Tuple[Dict, float, bool, Dict]:
        """Take a step by generating a token"""
        self.current_sequence.append(action)
        self.step_count += 1

        text = self.tokenizer.decode(self.current_sequence)

        # Termination conditions for code
        self.done = (
            self.step_count >= self.max_length or
            action == self.eos_token_id or
            '\n\n\n' in text[-20:]  # Multiple blank lines = code complete
        )

        if self.done:
            # Extract generated code (remove prompt)
            prompt_length = len(self.tokenizer.encode(self.current_problem['prompt']))
            generated_tokens = self.current_sequence[prompt_length:]
            generated_code = self.tokenizer.decode(generated_tokens)

            # Compute reward using dataset's evaluation
            reward = self.dataset.compute_reward(generated_code, self.current_problem)
        else:
            # Small step penalty
            reward = -0.01

        next_state = self.get_state()

        info = {
            'text': text,
            'length': len(self.current_sequence),
            'problem_id': self.current_problem['task_id']
        }

        return next_state, reward, self.done, info

    def get_state(self) -> Dict:
        return {
            'token_ids': torch.tensor(self.current_sequence),
            'step': self.step_count,
            'done': self.done
        }

## 5. Neural Network Architectures

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for transformers"""

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]

In [ ]:
class HierarchicalPolicy(nn.Module):
    """Two-level hierarchical policy for structured generation"""

    def __init__(self, vocab_size: int, d_model: int = 256, intention_dim: int = 64,
                 num_layers: int = 4, nhead: int = 4, max_len: int = 512):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.intention_dim = intention_dim
        self.max_len = max_len # Store max_len

        # Shared state encoder
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len) # Pass max_len

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            batch_first=True
        )
        self.state_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers // 2)

        # HIGH-LEVEL POLICY: State -> Intention distribution
        self.intention_mean = nn.Linear(d_model, intention_dim)
        self.intention_logstd = nn.Linear(d_model, intention_dim)

        # LOW-LEVEL POLICY: State + Intention -> Token distribution
        self.token_decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=d_model * 4,
                batch_first=True
            ),
            num_layers=num_layers // 2
        )

        self.intention_proj = nn.Linear(intention_dim, d_model)
        self.output_layer = nn.Linear(d_model, vocab_size)

        # Separate value heads
        self.high_value = nn.Linear(d_model, 1)
        self.low_value = nn.Linear(d_model, 1)

    def encode_state(self, token_ids):
        # Truncate input sequence to max_len for positional encoding
        seq_len = token_ids.size(1)
        if seq_len > self.max_len:
            token_ids = token_ids[:, :self.max_len]
            seq_len = self.max_len

        x = self.embedding(token_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.state_encoder(x)
        return x

    def sample_intention(self, state_encoding):
        pooled = state_encoding.mean(dim=1)
        mean = self.intention_mean(pooled)
        logstd = self.intention_logstd(pooled)
        std = torch.exp(logstd)
        dist = torch.distributions.Normal(mean, std)
        intention = dist.rsample()
        return intention, dist

    def forward(self, token_ids, intention=None, return_intention=False):
        state_encoding = self.encode_state(token_ids)

        if intention is None:
            intention, intention_dist = self.sample_intention(state_encoding)
        else:
            intention_dist = None

        intention_encoded = self.intention_proj(intention)
        intention_expanded = intention_encoded.unsqueeze(1).expand(
            -1, state_encoding.size(1), -1
        )

        # Ensure target sequence length for decoder matches encoder output length
        decoded = self.token_decoder(state_encoding, intention_expanded)
        logits = self.output_layer(decoded[:, -1, :])

        if return_intention:
            pooled = state_encoding.mean(dim=1)
            high_value = self.high_value(pooled)
            low_value = self.low_value(pooled)

            return {
                'logits': logits,
                'intention': intention,
                'intention_dist': intention_dist,
                'high_value': high_value,
                'low_value': low_value
            }
        else:
            return logits

    def get_action_distribution(self, token_ids, intention=None):
        logits = self.forward(token_ids, intention)
        return torch.distributions.Categorical(logits=logits)

## 6. Reward Functions

In [ ]:
class MultiComponentReward:
    """Multi-component reward system with fluency, coherence, task completion, and safety"""

    def __init__(self, tokenizer=None, task_evaluator=None):
        self.tokenizer = tokenizer
        self.task_evaluator = task_evaluator

        # Load GPT-2 for fluency scoring
        print("Loading GPT-2 for fluency evaluation...")
        self.fluency_device = device
        self.fluency_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.fluency_model = GPT2LMHeadModel.from_pretrained('gpt2').to(self.fluency_device)
        self.fluency_model.eval()

        if self.fluency_tokenizer.pad_token is None:
            self.fluency_tokenizer.pad_token = self.fluency_tokenizer.eos_token

        self.unsafe_patterns = [
            'kill', 'hate', 'violence', 'attack', 'bomb', 'weapon',
            'racist', 'sexist', 'discriminat'
        ]

    def compute_fluency(self, token_ids: torch.Tensor) -> float:
        if len(token_ids) < 2:
            return 0.0

        with torch.no_grad():
            if token_ids.dim() == 1:
                token_ids = token_ids.unsqueeze(0)
            token_ids = token_ids.to(self.fluency_device)
            outputs = self.fluency_model(token_ids, labels=token_ids)
            loss = outputs.loss.item()
            return -loss * 0.1

    def compute_coherence(self, token_ids: torch.Tensor, text: str = None) -> float:
        if len(token_ids) < 2:
            return 0.0

        token_list = token_ids.tolist() if torch.is_tensor(token_ids) else token_ids
        unique_tokens = len(set(token_list))
        total_tokens = len(token_list)
        diversity_ratio = unique_tokens / total_tokens if total_tokens > 0 else 0.0
        coherence_score = diversity_ratio * 0.2

        immediate_repeats = sum(1 for i in range(len(token_list) - 1)
                               if token_list[i] == token_list[i+1])
        repetition_penalty = -0.05 * immediate_repeats

        return coherence_score + repetition_penalty

    def compute_task_completion(self, token_ids: torch.Tensor, text: str = None,
                                prompt: str = None, target: str = None) -> float:
        if self.task_evaluator is not None:
            return self.task_evaluator(text, prompt, target)

        score = 0.0
        length = len(token_ids)
        if 5 <= length <= 100:
            score += 0.1
        elif length < 5:
            score -= 0.2
        elif length > 200:
            score -= 0.1

        return score

    def compute_safety(self, token_ids: torch.Tensor, text: str = None) -> float:
        if text is None and self.tokenizer is not None:
            text = self.tokenizer.decode(token_ids)

        if text is None:
            return 0.0

        text_lower = text.lower()
        for pattern in self.unsafe_patterns:
            if pattern in text_lower:
                return -1.0

        return 0.0

    def compute_reward(self, token_ids: torch.Tensor, text: str = None,
                      prompt: str = None, target: str = None) -> float:
        if text is None and self.tokenizer is not None:
            text = self.tokenizer.decode(token_ids)

        fluency = self.compute_fluency(token_ids)
        coherence = self.compute_coherence(token_ids, text)
        task_completion = self.compute_task_completion(token_ids, text, prompt, target)
        safety = self.compute_safety(token_ids, text)

        total_reward = (
            0.4 * fluency +
            0.2 * coherence +
            0.3 * task_completion +
            0.1 * safety
        )

        if safety < 0:
            total_reward += safety * 5.0

        return total_reward

## 7. Task-Specific Environments

In [ ]:
class QuestionAnsweringEnvironment:
    """Specialized environment for Q&A tasks"""

    def __init__(self, tokenizer, dataset=None, max_length: int = 100):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.max_length = max_length

        self.current_question = None
        self.expected_answer = None
        self.current_sequence = []
        self.step_count = 0
        self.done = False

        self.eos_token_id = tokenizer.eos_token_id

    def reset(self, question: str = None, answer: str = None) -> Dict:
        if question is None and self.dataset is not None:
            problem = self.dataset.get_random_problem()
            question = problem.get('question', problem.get('prompt', ''))
            answer = problem.get('answer', problem.get('expected', None))
        elif question is None:
            question = "What is 2+2?"
            answer = "4"

        self.current_question = question
        self.expected_answer = answer

        prompt_with_qa_format = f"Question: {question}\nAnswer:"
        token_ids = self.tokenizer.encode(prompt_with_qa_format, return_tensors='pt')[0]

        self.current_sequence = token_ids.tolist()
        self.step_count = 0
        self.done = False

        return self.get_state()

    def step(self, action: int) -> Tuple[Dict, float, bool, Dict]:
        self.current_sequence.append(action)
        self.step_count += 1

        text = self.tokenizer.decode(self.current_sequence)

        self.done = (
            self.step_count >= self.max_length or
            action == self.eos_token_id or
            '\n' in text[-5:]
        )

        if self.done:
            prompt_length = len(self.tokenizer.encode(f"Question: {self.current_question}\nAnswer:"))
            answer_tokens = self.current_sequence[prompt_length:]
            generated_answer = self.tokenizer.decode(answer_tokens).strip()
            reward = self._compute_qa_reward(generated_answer)
        else:
            reward = -0.01

        next_state = self.get_state()
        info = {
            'text': text,
            'question': self.current_question,
            'expected_answer': self.expected_answer,
            'length': len(self.current_sequence)
        }

        return next_state, reward, self.done, info

    def _compute_qa_reward(self, generated_answer: str) -> float:
        reward = 0.0

        if self.expected_answer is not None:
            expected_lower = self.expected_answer.lower().strip()
            generated_lower = generated_answer.lower().strip()

            if expected_lower == generated_lower:
                reward += 10.0
            elif expected_lower in generated_lower:
                reward += 7.0
            elif generated_lower in expected_lower:
                reward += 5.0
            else:
                reward -= 1.0

        answer_length = len(generated_answer.split())
        if answer_length < 1:
            reward -= 5.0
        elif 1 <= answer_length <= 10:
            reward += 1.0
        elif answer_length > 50:
            reward -= 2.0

        return reward

    def get_state(self) -> Dict:
        return {
            'token_ids': torch.tensor(self.current_sequence),
            'step': self.step_count,
            'done': self.done,
            'question': self.current_question
        }

## 8. Rollout Buffer

In [ ]:
class RolloutBuffer:
    """Experience buffer for storing and processing trajectories"""

    def __init__(self):
        self.clear()

    def clear(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
        self.values = []
        self.dones = []
        self.intentions = []

    def add(self, state, action, reward, log_prob, value, done, intention=None):
        self.states.append(state['token_ids'])
        self.actions.append(action)
        self.rewards.append(reward)
        self.log_probs.append(log_prob)
        self.values.append(value)
        self.dones.append(done)

        if intention is not None:
            self.intentions.append(intention)

    def get_batches(self, gamma=0.99, gae_lambda=0.95):
        rewards = np.array(self.rewards)
        values = np.array([v.item() if torch.is_tensor(v) else v for v in self.values])
        dones = np.array(self.dones, dtype=np.float32)

        # Compute advantages using GAE
        advantages = np.zeros_like(rewards)
        last_advantage = 0

        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_value = 0
            else:
                next_value = values[t + 1]

            delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
            advantages[t] = last_advantage = delta + gamma * gae_lambda * (1 - dones[t]) * last_advantage

        returns = advantages + values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # Pad states
        max_len = max(len(s) for s in self.states)
        padded_states = []
        for s in self.states:
            if len(s) < max_len:
                pad_len = max_len - len(s)
                s_padded = torch.cat([s, torch.zeros(pad_len, dtype=s.dtype)])
            else:
                s_padded = s
            padded_states.append(s_padded)

        batch = {
            'states': torch.stack(padded_states),
            'actions': torch.tensor(self.actions, dtype=torch.long),
            'log_probs': torch.tensor(self.log_probs, dtype=torch.float32),
            'returns': torch.tensor(returns, dtype=torch.float32),
            'advantages': torch.tensor(advantages, dtype=torch.float32),
        }

        if len(self.intentions) > 0:
            batch['intentions'] = torch.stack(self.intentions)

        return batch

    def __len__(self):
        return len(self.rewards)

## 9. Hierarchical PPO Trainer

In [ ]:
class HierarchicalPPOTrainer:
    """PPO trainer for hierarchical two-level policy"""

    def __init__(self, hierarchical_policy, lr: float = 3e-4, clip_ratio: float = 0.2,
                 value_coef: float = 0.5, entropy_coef: float = 0.01, intention_coef: float = 0.1):
        self.policy = hierarchical_policy
        self.clip_ratio = clip_ratio
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.intention_coef = intention_coef

        self.optimizer = optim.Adam(hierarchical_policy.parameters(), lr=lr)
        self.device = next(hierarchical_policy.parameters()).device

    def compute_hierarchical_policy_loss(self, states, actions, old_log_probs,
                                         old_intentions, advantages):
        outputs = self.policy(states, return_intention=True)

        # Low-level policy loss
        action_dist = torch.distributions.Categorical(logits=outputs['logits'])
        new_log_probs = action_dist.log_prob(actions)

        ratio = torch.exp(new_log_probs - old_log_probs)
        clipped_ratio = torch.clamp(ratio, 1 - self.clip_ratio, 1 + self.clip_ratio)

        low_level_loss = -torch.min(
            ratio * advantages,
            clipped_ratio * advantages
        ).mean()

        token_entropy = action_dist.entropy().mean()

        # High-level policy loss
        if outputs['intention_dist'] is not None:
            old_intention_log_prob = outputs['intention_dist'].log_prob(old_intentions).sum(dim=-1)
            intention_entropy = outputs['intention_dist'].entropy().sum(dim=-1).mean()
            high_level_loss = -old_intention_log_prob.mean()
        else:
            high_level_loss = torch.tensor(0.0, device=self.device)
            intention_entropy = torch.tensor(0.0, device=self.device)

        policy_loss = low_level_loss + self.intention_coef * high_level_loss
        total_entropy = token_entropy + 0.1 * intention_entropy

        return policy_loss, total_entropy, high_level_loss

    def compute_hierarchical_value_loss(self, states, returns):
        outputs = self.policy(states, return_intention=True)

        high_value = outputs['high_value'].squeeze(-1)
        low_value = outputs['low_value'].squeeze(-1)

        high_value_loss = nn.functional.mse_loss(high_value, returns)
        low_value_loss = nn.functional.mse_loss(low_value, returns)

        return (high_value_loss + low_value_loss) / 2.0

    def update(self, buffer, epochs: int = 4):
        batches = buffer.get_batches()

        states = batches['states'].to(self.device)
        actions = batches['actions'].to(self.device)
        old_log_probs = batches['log_probs'].to(self.device)
        returns = batches['returns'].to(self.device)
        advantages = batches['advantages'].to(self.device)

        if 'intentions' in batches:
            old_intentions = batches['intentions'].to(self.device)
        else:
            with torch.no_grad():
                outputs = self.policy(states, return_intention=True)
                old_intentions = outputs['intention']

        stats = {
            'policy_loss': 0.0,
            'value_loss': 0.0,
            'entropy': 0.0,
            'intention_loss': 0.0
        }

        for epoch in range(epochs):
            policy_loss, entropy, intention_loss = self.compute_hierarchical_policy_loss(
                states, actions, old_log_probs, old_intentions, advantages
            )
            value_loss = self.compute_hierarchical_value_loss(states, returns)

            total_loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy

            self.optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
            self.optimizer.step()

            stats['policy_loss'] += policy_loss.item()
            stats['value_loss'] += value_loss.item()
            stats['entropy'] += entropy.item()
            stats['intention_loss'] += intention_loss.item()

        for key in stats:
            stats[key] /= epochs

        return stats

    def collect_episode_with_intentions(self, env, buffer, max_steps: int = 100):
        state = env.reset()
        done = False
        episode_reward = 0.0
        steps = 0

        while not done and steps < max_steps:
            token_ids = state['token_ids'].unsqueeze(0).to(self.device)

            with torch.no_grad():
                outputs = self.policy(token_ids, return_intention=True)
                action_dist = torch.distributions.Categorical(logits=outputs['logits'])
                action = action_dist.sample()
                log_prob = action_dist.log_prob(action)
                intention = outputs['intention']
                value = outputs['low_value']

            next_state, reward, done, info = env.step(action.item())

            buffer.add(
                state=state,
                action=action.item(),
                reward=reward,
                log_prob=log_prob.item(),
                value=value.item(),
                done=done,
                intention=intention.cpu()
            )

            state = next_state
            episode_reward += reward
            steps += 1

        return episode_reward

## 10. Initialize Components

In [ ]:
# # Load tokenizer
# print("Loading tokenizer...")
# tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
# tokenizer.pad_token = tokenizer.eos_token

# # Create dataset
# print("Creating dataset...")
# dataset = TinyDataset()
# print(f"Dataset size: {len(dataset)}")

# # Create environment
# print("Creating Q&A environment...")
# env = QuestionAnsweringEnvironment(
#     tokenizer=tokenizer,
#     dataset=dataset,
#     max_length=100
# )

# # Create reward function
# print("Creating multi-component reward function...")
# reward_fn = MultiComponentReward(tokenizer=tokenizer)

# print("\n✓ All components initialized!")

In [ ]:
# CELL 10 (REPLACEMENT): Initialize Components for Code Generation

# Configuration - Choose your dataset
DATASET_TYPE = 'humaneval'  # Options: 'humaneval', 'stack', 'codechain', 'redpajama'
MAX_LENGTH = 512  # Longer for code generation - Aligned with PositionalEncoding max_len

print(f"Initializing components for {DATASET_TYPE.upper()} dataset...")
print("="*70)

# Load tokenizer
print("\n1. Loading tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Create code dataset
print(f"\n2. Creating {DATASET_TYPE} dataset...")
if DATASET_TYPE == 'humaneval':
    dataset = create_code_dataset('humaneval', use_subset=3)  # Start with subset
elif DATASET_TYPE == 'stack':
    dataset = create_code_dataset('stack', language='python', num_samples=100)
elif DATASET_TYPE == 'codechain':
    dataset = create_code_dataset('codechain', num_samples=100)
elif DATASET_TYPE == 'redpajama':
    dataset = create_code_dataset('redpajama', num_samples=100)

print(f"   Dataset size: {len(dataset)}")

# Create code generation environment
print("\n3. Creating Code Generation environment...")
env = CodeGenerationEnvironment(
    tokenizer=tokenizer,
    dataset=dataset,
    max_length=MAX_LENGTH # Use the consistent MAX_LENGTH
)

# Create multi-component reward function
print("\n4. Creating multi-component reward function...")
reward_fn = MultiComponentReward(tokenizer=tokenizer)

print("\n✓ All components initialized for code generation!")
print("="*70)

Initializing components for HUMANEVAL dataset...

1. Loading tokenizer...

2. Creating humaneval dataset...
Loading HumanEval dataset...
Loaded 20 code problems
   Dataset size: 20

3. Creating Code Generation environment...

4. Creating multi-component reward function...
Loading GPT-2 for fluency evaluation...

✓ All components initialized for code generation!


## 11. Create Hierarchical Policy

In [ ]:
# Create hierarchical policy
print("Creating hierarchical policy...")
vocab_size = tokenizer.vocab_size

policy = HierarchicalPolicy(
    vocab_size=vocab_size,
    d_model=256,
    intention_dim=64,
    num_layers=4,
    nhead=4
).to(device)

print(f"Policy created with {sum(p.numel() for p in policy.parameters()):,} parameters")

# Create trainer
print("Creating hierarchical PPO trainer...")
trainer = HierarchicalPPOTrainer(policy, lr=3e-4)

print("\n✓ Policy and trainer ready!")

Creating hierarchical policy...
Policy created with 29,518,291 parameters
Creating hierarchical PPO trainer...

✓ Policy and trainer ready!


## 12. Training Loop

In [ ]:
# CELL 12 (REPLACEMENT): Training Configuration for Code Generation

# Training configuration
NUM_ITERATIONS = 100  # More iterations for code tasks
EPISODES_PER_ITERATION = 1  # Fewer episodes (code generation is slower)
MAX_LENGTH = 112  # Longer sequences for code - Aligned with PositionalEncoding max_len
LOG_INTERVAL = 20  # Log less frequently

print(f"\n{'='*70}")
print("Starting Hierarchical Policy Training on Code Generation")
print(f"{'='*70}")
print(f"Dataset: {DATASET_TYPE.upper()}")
print(f"Iterations: {NUM_ITERATIONS}")
print(f"Episodes per iteration: {EPISODES_PER_ITERATION}")
print(f"Max length: {MAX_LENGTH}")
print(f"{'='*70}\n")

best_reward = float('-inf')
training_history = []

for iteration in tqdm(range(NUM_ITERATIONS), desc="Training"):
    policy.train()
    buffer = RolloutBuffer()
    episode_rewards = []

    # Collect episodes
    for episode in range(EPISODES_PER_ITERATION):
        episode_reward = trainer.collect_episode_with_intentions(
            env=env,
            buffer=buffer,
            max_steps=MAX_LENGTH # Use the consistent MAX_LENGTH
        )
        episode_rewards.append(episode_reward)

    # Update policy
    if len(buffer) > 0:
        stats = trainer.update(buffer, epochs=4)
    else:
        stats = {}

    # Logging
    avg_reward = sum(episode_rewards) / len(episode_rewards)
    training_history.append(avg_reward)

    if (iteration + 1) % LOG_INTERVAL == 0:
        print(f"\n{'='*70}")
        print(f"Iteration {iteration + 1}/{NUM_ITERATIONS}")
        print(f"{'='*70}")
        print(f"  Avg Training Reward: {avg_reward:.2f}")
        print(f"  Policy Loss: {stats.get('policy_loss', 0):.4f}")
        print(f"  Value Loss: {stats.get('value_loss', 0):.4f}")
        print(f"  Entropy: {stats.get('entropy', 0):.4f}")
        print(f"  Intention Loss: {stats.get('intention_loss', 0):.4f}")

        # Generate sample code
        policy.eval()
        state = env.reset()
        done = False
        steps = 0

        while not done and steps < 100:
            token_ids = state['token_ids'].unsqueeze(0).to(device)
            with torch.no_grad():
                outputs = policy(token_ids, return_intention=True)
                action_dist = torch.distributions.Categorical(logits=outputs['logits'])
                action = action_dist.sample()
            state, _, done, info = env.step(action.item())
            steps += 1

        print(f"\n  Sample Generated Code:")
        print(f"  {'-'*66}")
        # Show first 200 chars of generated code
        code_snippet = info['text'][:200].replace('\n', '\n  ')
        print(f"  {code_snippet}...")
        print(f"  {'-'*66}")

    # Save best model
    if avg_reward > best_reward:
        best_reward = avg_reward
        print(f"\n  ✓ New best model! Reward: {best_reward:.2f}")

print(f"\n{'='*70}")
print("Training Complete!")
print(f"{'='*70}")
print(f"Best reward achieved: {best_reward:.2f}")


Starting Hierarchical Policy Training on Code Generation
Dataset: HUMANEVAL
Iterations: 500
Episodes per iteration: 5
Max length: 512



Training:   0%|          | 0/500 [00:14<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.00 GiB. GPU 0 has a total capacity of 79.32 GiB of which 2.30 GiB is free. Process 20039 has 77.01 GiB memory in use. Of the allocated memory 74.70 GiB is allocated by PyTorch, and 1.82 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 13. Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(training_history)
plt.xlabel('Iteration')
plt.ylabel('Average Reward')
plt.title('Training Progress: Hierarchical Policy on Q&A Task')
plt.grid(True, alpha=0.3)
plt.show()

# Compute moving average
window = 10
if len(training_history) >= window:
    moving_avg = np.convolve(training_history, np.ones(window)/window, mode='valid')
    plt.figure(figsize=(12, 6))
    plt.plot(training_history, alpha=0.3, label='Raw')
    plt.plot(range(window-1, len(training_history)), moving_avg, label=f'{window}-iteration Moving Average')
    plt.xlabel('Iteration')
    plt.ylabel('Average Reward')
    plt.title('Training Progress with Moving Average')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 14. Final Evaluation

In [ ]:
def evaluate_policy(policy, env, tokenizer, num_episodes=10):
    """Evaluate trained policy"""
    policy.eval()

    total_reward = 0.0
    successful_episodes = 0
    generations = []

    for _ in range(num_episodes):
        state = env.reset()
        done = False
        episode_reward = 0.0

        while not done:
            token_ids = state['token_ids'].unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = policy(token_ids, return_intention=True)
                action_dist = torch.distributions.Categorical(logits=outputs['logits'])
                action = action_dist.sample()

            state, reward, done, info = env.step(action.item())
            episode_reward += reward

        total_reward += episode_reward
        if episode_reward > 0:
            successful_episodes += 1

        generations.append(info['text'])

    return {
        'avg_reward': total_reward / num_episodes,
        'success_rate': successful_episodes / num_episodes,
        'generations': generations
    }

print(f"\n{'='*70}")
print("Final Evaluation")
print(f"{'='*70}\n")

eval_results = evaluate_policy(policy, env, tokenizer, num_episodes=10)

print(f"Average Reward: {eval_results['avg_reward']:.2f}")
print(f"Success Rate: {eval_results['success_rate']:.1%}")
print(f"\nSample Generations:\n")

for i, gen in enumerate(eval_results['generations'][:5]):
    print(f"{i+1}. {gen}")
    print()

## 15. Test Specific Questions

In [ ]:
def test_question(policy, tokenizer, question, max_steps=50):
    """Test policy on a specific question"""
    policy.eval()

    # Create temporary environment for this question
    test_env = QuestionAnsweringEnvironment(tokenizer, dataset=None, max_length=100)
    state = test_env.reset(question=question)

    done = False
    steps = 0

    while not done and steps < max_steps:
        token_ids = state['token_ids'].unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = policy(token_ids, return_intention=True)
            action_dist = torch.distributions.Categorical(logits=outputs['logits'])
            action = action_dist.sample()

        state, reward, done, info = test_env.step(action.item())
        steps += 1

    return info['text']

# Test on custom questions
test_questions = [
    "What is 7+3?",
    "What is 12-5?",
    "What is the capital of France?",
    "Who invented the telephone?",
    "What color is the sky?"
]

print(f"\n{'='*70}")
print("Testing on Custom Questions")
print(f"{'='*70}\n")

for question in test_questions:
    answer = test_question(policy, tokenizer, question)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

In [ ]:
# NEW CELL: Test Code Generation

def test_code_generation(policy, tokenizer, dataset, num_samples=5):
    """Test policy on code generation tasks"""
    policy.eval()

    print(f"\n{'='*70}")
    print(f"Testing Code Generation on {num_samples} problems")
    print(f"{'='*70}\n")

    for i in range(num_samples):
        problem = dataset.get_random_problem()

        # Create temp environment
        temp_env = CodeGenerationEnvironment(tokenizer, dataset, max_length=512)
        temp_env.current_problem = problem

        # Generate
        prompt = problem['prompt']
        token_ids = tokenizer.encode(prompt, return_tensors='pt')[0]
        state = {'token_ids': token_ids, 'step': 0, 'done': False}

        done = False
        steps = 0

        while not done and steps < 200:
            token_ids = state['token_ids'].unsqueeze(0).to(device)
            with torch.no_grad():
                outputs = policy(token_ids, return_intention=True)
                action_dist = torch.distributions.Categorical(logits=outputs['logits'])
                action = action_dist.sample()

            # Manual step
            current_seq = state['token_ids'].tolist()
            current_seq.append(action.item())
            text = tokenizer.decode(current_seq)

            done = steps >= 200 or action.item() == tokenizer.eos_token_id
            state = {'token_ids': torch.tensor(current_seq), 'step': steps, 'done': done}
            steps += 1

        # Extract generated code
        prompt_len = len(tokenizer.encode(prompt))
        generated_code = tokenizer.decode(state['token_ids'][prompt_len:])

        # Show results
        print(f"Problem {i+1}: {problem['task_id']}")
        print(f"Prompt:\n{prompt[:150]}...")
        print(f"\nGenerated Code:\n{generated_code[:300]}")
        print(f"\n{'-'*70}\n")

# Run test
test_code_generation(policy, tokenizer, dataset, num_samples=3)

## 16. Save Model

In [ ]:
# Save the trained model
save_path = 'hierarchical_policy_qa.pt'

torch.save({
    'policy_state_dict': policy.state_dict(),
    'best_reward': best_reward,
    'training_history': training_history,
    'vocab_size': vocab_size,
    'd_model': 256,
    'intention_dim': 64,
    'num_layers': 4,
    'nhead': 4
}, save_path)

print(f"✓ Model saved to {save_path}")

# Download to local machine
from google.colab import files
files.download(save_path)
print(f"✓ Model downloaded!")

## 17. Load Saved Model (Optional)

In [ ]:
# To load a saved model
def load_model(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Recreate model with saved architecture
    loaded_policy = HierarchicalPolicy(
        vocab_size=checkpoint['vocab_size'],
        d_model=checkpoint['d_model'],
        intention_dim=checkpoint['intention_dim'],
        num_layers=checkpoint['num_layers'],
        nhead=checkpoint['nhead']
    ).to(device)

    # Load weights
    loaded_policy.load_state_dict(checkpoint['policy_state_dict'])
    loaded_policy.eval()

    print(f"✓ Model loaded from {checkpoint_path}")
    print(f"  Best reward: {checkpoint['best_reward']:.2f}")

    return loaded_policy

# Example usage:
# loaded_policy = load_model('hierarchical_policy_qa.pt')

## Summary

This notebook implements a complete hierarchical RL-based language model:

**Key Components:**
1. ✓ Multi-component reward system (fluency, coherence, task completion, safety)
2. ✓ Task-specific Q&A environment
3. ✓ Hierarchical policy with two levels:
   - High-level: Semantic intentions
   - Low-level: Token generation
4. ✓ Hierarchical PPO training algorithm
5. ✓ Evaluation and testing tools

**Training Features:**
- Progressive training with GAE (Generalized Advantage Estimation)
- Gradient clipping for stability
- Intention regularization for coherence
- Dual value functions (high-level and low-level)

**Next Steps:**
- Train for more iterations
- Try different datasets (math, conversation)
- Experiment with hyperparameters
- Implement energy-based value functions (Week 7-8)
